# 01 — lab value (experiment)

This notebook is **not** the live website. The site already shows a lab index, not a transfer fee.

Plan: pull stats you are allowed to use (goals, assists, minutes, position; age only if the API has it). Pull **listed** transfer fees only if football-data.org (or another licensed source) actually has them. Train linear regression **only on rows that have a fee**. Plot predicted vs actual. Write the MAE in a sentence.

If the fee column is empty, **stop**. Do not scrape Transfermarkt. Do not invent Y.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv(Path(".env"))
TOKEN = os.getenv("FOOTBALL_DATA_TOKEN", "").strip()
PL_COMPETITION_ID = 2021  # Premier League on football-data.org
print("Token loaded:" , bool(TOKEN))
if not TOKEN:
    print("Paste FOOTBALL_DATA_TOKEN into lab-models/.env then re-run. Get a free token at football-data.org.")

## Docs to read before you call the API

- What you may call on the free tier
- Rate limits (do not hammer)
- Premier League competition id: `2021`
- Whether **listed transfer fees** exist as a field. If they do not, leave the column empty and skip the regression.

In [ ]:
import requests
import pandas as pd

def fd_get(path, params=None):
    if not TOKEN:
        raise SystemExit("No token yet. Fill lab-models/.env first.")
    url = "https://api.football-data.org/v4" + path
    response = requests.get(
        url,
        headers={"X-Auth-Token": TOKEN},
        params=params or {},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

# Starter call: competition info. Next sessions: scorers / teams / whatever the docs allow.
if TOKEN:
    info = fd_get(f"/competitions/{PL_COMPETITION_ID}")
    print(info.get("name"), info.keys()))
else:
    print("Skip network until .env has a token.")

In [ ]:
# Build a table: player, position, goals, assists, minutes, age (if present), listed_fee (if present).
# If listed_fee is missing for everyone, leave it empty and do not train.

players = pd.DataFrame(
    columns=["player", "position", "goals", "assists", "minutes", "age", "listed_fee"]
)
players

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

with_fee = players.dropna(subset=["listed_fee"]) if not players.empty else players
if with_fee.empty:
    print("No listed fees. Stop here. Keep the website lab index. This is not a scout's valuation.")
else:
    features = ["goals", "assists", "minutes"]
    model = LinearRegression()
    model.fit(with_fee[features], with_fee["listed_fee"])
    pred = model.predict(with_fee[features])
    mae = mean_absolute_error(with_fee["listed_fee"], pred)
    print(f"MAE on rows that have a listed fee: {mae:.0f}. This is not a scout's valuation.")
    plt.scatter(with_fee["listed_fee"], pred)
    plt.xlabel("Listed fee")
    plt.ylabel("Predicted")
    plt.title("Lab model vs listed fee (not a valuation)")
    plt.show()

**This is not a scout’s valuation.**

Later, on Scout, show “model £X vs listed £Y” **only when Y exists**.